# Stage 02 · Step 3 — Surrogate-driven τ search

Use the SSL+RUL model to predict component-level remaining life on observed telemetry, derive an expected number of preventive / corrective events for any candidate τ schedule, and let Optuna optimise τ over that surrogate without paying the simulator's full cost.

The surrogate's predictions are validated against the **real simulator** at the end: the top-K τ vectors found by the surrogate are re-simulated with `lib.env_runner.run_with_tau` and scored with `lib.objective.scalar_objective`, then compared against Stage 01's leaderboard.

If the surrogate is well-calibrated, Stage 02 returns a τ vector whose true cost is within a few percent of Stage 01's optimum — at a fraction of the wall-clock time spent inside the optimiser.

In [1]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader
from transformers import PatchTSTConfig, PatchTSTForRegression

from ml_models.lib.data import (
    DEFAULT_FLEET_PATH,
    TEST_PRINTERS,
    TRAIN_PRINTERS,
    VAL_PRINTERS,
    filter_printers,
    load_fleet,
)
from ml_models.lib.env_runner import default_dates, run_with_tau
from ml_models.lib.features import build_feature_matrix
from ml_models.lib.objective import scalar_objective
from ml_models import PROJECT_ROOT
from sdg.generate import build_printer_city_map, load_configs
from sdg.schema import COMPONENT_IDS

MODELS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/models'
RESULTS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
with (MODELS_DIR / 'ssl_config.json').open() as handle:
    saved = json.load(handle)
patch_cfg = PatchTSTConfig(**saved['patch_cfg'])
patch_cfg.num_targets = 6
patch_cfg.prediction_length = 1
patch_cfg.use_cls_token = False
model = PatchTSTForRegression(patch_cfg).to(device)
model.load_state_dict(torch.load(MODELS_DIR / 'rul_head_ssl.pt', map_location=device))
model.eval()

scaler = np.load(MODELS_DIR / 'feature_scaler.npz', allow_pickle=True)
channel_mean = scaler['mean']; channel_std = scaler['std']
CONTEXT_LENGTH = int(saved['train_cfg']['context_length'])
print('Loaded RUL model. Context length:', CONTEXT_LENGTH)

[transformers] Setting `do_mask_input` parameter to False.


Loaded RUL model. Context length: 360


In [3]:
fleet = load_fleet(DEFAULT_FLEET_PATH)
feat_fleet, feature_cols = build_feature_matrix(fleet)

def predict_rul_for_window(printer_id: int, end_day: int) -> np.ndarray:
    df = filter_printers(feat_fleet, [printer_id])
    df = df.sort_values('day')
    arr = df[feature_cols].to_numpy(dtype=np.float32)
    if end_day < CONTEXT_LENGTH:
        return np.full(6, np.nan, dtype=np.float32)
    window = arr[end_day - CONTEXT_LENGTH:end_day]
    normed = (window - channel_mean) / channel_std
    x = torch.from_numpy(normed).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(past_values=x).regression_outputs.squeeze(-1).view(-1)
    return (out.cpu().numpy() * 365.0).clip(0.0, 365.0)

sample_rul = predict_rul_for_window(VAL_PRINTERS[0], end_day=2000)
print('Predicted RUL for printer', VAL_PRINTERS[0], 'at day 2000:', sample_rul)

Predicted RUL for printer 70 at day 2000: [nan nan nan nan nan nan]


## Surrogate cost model

For each candidate τ vector and each held-out printer, estimate annual preventive and corrective events using:

$$n_{prev,i} \approx H \cdot (24 / \tau_i) \qquad n_{corr,i} \approx H \cdot \hat{\lambda}_{cm,i}(\tau_i)$$

where $H$ is the simulated horizon (years), and $\hat{\lambda}_{cm,i}$ is a smooth function of $\tau_i$ fit from the RUL predictions: rapidly decreasing for short τ (frequent maintenance prevents failures), increasing for long τ (more failures slip through). The simplest such fit uses a single observation: at the baseline τ, observed corrective events are known from `fleet_baseline.parquet`; at τ→∞, every component eventually fails and contributes a corrective event per nominal life. We interpolate between these regimes via the predicted RUL on validation printers.

In [4]:
components_cfg, couplings_cfg, cities_cfg = load_configs()
components = components_cfg['components']
DATES = default_dates()
HORIZON_YEARS = len(DATES) / 365.25

anchor_pm: dict[str, int] = {}
anchor_cm: dict[str, int] = {}
for component_id in COMPONENT_IDS:
    anchor_pm[component_id] = int(filter_printers(fleet, list(TRAIN_PRINTERS) + list(VAL_PRINTERS))[f'maint_{component_id}'].sum())
    anchor_cm[component_id] = int(filter_printers(fleet, list(TRAIN_PRINTERS) + list(VAL_PRINTERS))[f'failure_{component_id}'].sum())
anchor_tau = {component_id: float(components[component_id]['tau_nom_d']) for component_id in COMPONENT_IDS}
anchor_pm, anchor_cm, anchor_tau

({'C1': 0, 'C2': 181, 'C3': 455, 'C4': 86, 'C5': 1325, 'C6': 591},
 {'C1': 273490, 'C2': 3495, 'C3': 257798, 'C4': 96420, 'C5': 610, 'C6': 357},
 {'C1': 25.0,
  'C2': 166.6666667,
  'C3': 7.0,
  'C4': 41.6666667,
  'C5': 166.6666667,
  'C6': 333.3333333})

In [5]:
def surrogate_score(tau_vector: dict[str, float]) -> dict:
    # event counts pool TRAIN+VAL, so normalise to the same denominator
    n_printers = len(TRAIN_PRINTERS) + len(VAL_PRINTERS)
    pm_total = 0.0
    cm_total = 0.0
    downtime_total = 0.0
    for component_id in COMPONENT_IDS:
        spec = components[component_id]
        tau_new = float(tau_vector[component_id])
        tau_base = anchor_tau[component_id]
        ratio = tau_base / tau_new if tau_new > 0 else 1.0
        # Linear scaling of preventive events with maintenance frequency.
        pm_count = anchor_pm[component_id] * ratio
        # Failure rate inflates with τ: when maintenance is rarer, failures grow.
        cm_count = anchor_cm[component_id] * (tau_new / tau_base) ** float(spec.get('b_M', 1.5))
        # Add an upper bound: at most one corrective per nominal life per printer.
        cm_cap = n_printers * (HORIZON_YEARS * 365.25 / float(spec['L_nom_d']))
        cm_count = min(cm_count, cm_cap)
        pm_total += pm_count * float(spec['cost_preventive_eur'])
        cm_total += cm_count * float(spec['cost_corrective_eur'])
        downtime_total += pm_count * float(spec['downtime_preventive_d']) * 24
        downtime_total += cm_count * float(spec['downtime_corrective_d']) * 24
    norm = n_printers * HORIZON_YEARS
    annual_cost = (pm_total + cm_total) / norm
    total_hours = len(DATES) * 24.0 * n_printers
    availability = (total_hours - downtime_total) / total_hours if total_hours > 0 else 1.0
    deficit = max(0.0, 0.95 - availability)
    return {
        'value': annual_cost + 1_000_000.0 * deficit,
        'annual_cost': annual_cost,
        'availability': availability,
    }

surrogate_score({c: anchor_tau[c] for c in COMPONENT_IDS})

{'value': 162073.9121392074,
 'annual_cost': 162073.9121392074,
 'availability': 0.9541695971497193}

In [6]:
TAU_RANGES = {
    'C1': (2.08, 83.33),
    'C2': (20.83, 833.33),
    'C3': (1.0, 20.83),
    'C4': (4.17, 83.33),
    'C5': (20.83, 333.33),
    'C6': (41.67, 833.33),
}

def trial_to_tau(trial: optuna.Trial) -> dict[str, float]:
    return {c: trial.suggest_float(f'tau_{c}', low, high, log=True) for c, (low, high) in TAU_RANGES.items()}

def surrogate_objective(trial: optuna.Trial) -> float:
    tau_vector = trial_to_tau(trial)
    score = surrogate_score(tau_vector)
    for key in ('annual_cost', 'availability'):
        trial.set_user_attr(key, float(score[key]))
    return float(score['value'])

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=99),
)
study.optimize(surrogate_objective, n_trials=500, show_progress_bar=True)
study_df = study.trials_dataframe().sort_values('value').reset_index(drop=True)
study_df.head(5)

[I 2026-04-25 19:15:13,543] A new study created in memory with name: no-name-2489cf90-ef7e-4a81-b9a7-cf5224090008


  0%|          | 0/500 [00:00<?, ?it/s]

[I 2026-04-25 19:15:13,546] Trial 0 finished with value: 163958.90106444026 and parameters: {'tau_C1': 24.862784114280885, 'tau_C2': 126.08203678755811, 'tau_C3': 12.262291285523435, 'tau_C4': 4.581812165424144, 'tau_C5': 195.76318491745727, 'tau_C6': 226.823525354764}. Best is trial 0 with value: 163958.90106444026.
[I 2026-04-25 19:15:13,546] Trial 1 finished with value: 160775.20771084604 and parameters: {'tau_C1': 6.238414643049031, 'tau_C2': 24.74581600388065, 'tau_C3': 20.245556651720225, 'tau_C4': 4.256121813156558, 'tau_C5': 176.060766994472, 'tau_C6': 390.26774322218074}. Best is trial 1 with value: 160775.20771084604.
[I 2026-04-25 19:15:13,547] Trial 2 finished with value: 161984.6944858456 and parameters: {'tau_C1': 8.375275052139996, 'tau_C2': 128.93671885260628, 'tau_C3': 16.787818954695982, 'tau_C4': 13.6297979806455, 'tau_C5': 310.1079943485298, 'tau_C6': 200.48581254087497}. Best is trial 1 with value: 160775.20771084604.
[I 2026-04-25 19:15:13,548] Trial 3 finished wi

[I 2026-04-25 19:15:13,749] Trial 80 finished with value: 162862.96698510245 and parameters: {'tau_C1': 2.613608975138153, 'tau_C2': 131.47384349301268, 'tau_C3': 5.083090554798016, 'tau_C4': 18.375554311589553, 'tau_C5': 291.85608401227466, 'tau_C6': 196.1538960448892}. Best is trial 51 with value: 151565.3889593649.
[I 2026-04-25 19:15:13,752] Trial 81 finished with value: 151286.48857687743 and parameters: {'tau_C1': 2.0932609261462307, 'tau_C2': 67.50851177529745, 'tau_C3': 4.927269035812098, 'tau_C4': 24.677046573165992, 'tau_C5': 106.10827956191999, 'tau_C6': 147.732944813045}. Best is trial 81 with value: 151286.48857687743.
[I 2026-04-25 19:15:13,755] Trial 82 finished with value: 151226.7445724159 and parameters: {'tau_C1': 2.0982712549810882, 'tau_C2': 78.31065723840558, 'tau_C3': 5.859392742550544, 'tau_C4': 21.465785461026808, 'tau_C5': 106.73094181756605, 'tau_C6': 141.8526911559832}. Best is trial 82 with value: 151226.7445724159.
[I 2026-04-25 19:15:13,758] Trial 83 fini

[I 2026-04-25 19:15:13,953] Trial 138 finished with value: 150943.12893235614 and parameters: {'tau_C1': 2.0897619405182435, 'tau_C2': 34.603434077174235, 'tau_C3': 5.293641955770156, 'tau_C4': 18.78246939473194, 'tau_C5': 116.12052447371948, 'tau_C6': 82.71581507025462}. Best is trial 123 with value: 148437.70842425028.
[I 2026-04-25 19:15:13,957] Trial 139 finished with value: 153966.11407578486 and parameters: {'tau_C1': 2.239329031473613, 'tau_C2': 32.179171589136054, 'tau_C3': 5.255326016845416, 'tau_C4': 17.04145663683287, 'tau_C5': 118.26510589714199, 'tau_C6': 79.41662711837053}. Best is trial 123 with value: 148437.70842425028.
[I 2026-04-25 19:15:13,960] Trial 140 finished with value: 162427.01761370813 and parameters: {'tau_C1': 2.9761481258428875, 'tau_C2': 34.24691400240797, 'tau_C3': 6.005948849726919, 'tau_C4': 18.32933281905218, 'tau_C5': 100.48764000897539, 'tau_C6': 66.52362766415031}. Best is trial 123 with value: 148437.70842425028.
[I 2026-04-25 19:15:13,964] Trial

...

[I 2026-04-25 19:15:14,973] Trial 376 finished with value: 149388.71268104893 and parameters: {'tau_C1': 2.2325034336699687, 'tau_C2': 24.89162361042386, 'tau_C3': 19.61015122730292, 'tau_C4': 34.185058573643545, 'tau_C5': 40.338799801595094, 'tau_C6': 643.2469487096346}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:14,977] Trial 377 finished with value: 146677.801856611 and parameters: {'tau_C1': 2.082776152672594, 'tau_C2': 22.194966476809537, 'tau_C3': 17.89873121703667, 'tau_C4': 40.804471656099295, 'tau_C5': 36.755923914125624, 'tau_C6': 724.2958224140282}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:14,982] Trial 378 finished with value: 157616.92124297775 and parameters: {'tau_C1': 2.689873943267878, 'tau_C2': 20.896998815469985, 'tau_C3': 18.313046651329138, 'tau_C4': 43.88640655386434, 'tau_C5': 38.29883653070965, 'tau_C6': 781.6936640688092}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:14,987] Trial 

[I 2026-04-25 19:15:15,175] Trial 415 finished with value: 145445.67869256876 and parameters: {'tau_C1': 2.089137529473507, 'tau_C2': 26.449445987980162, 'tau_C3': 17.467591749597034, 'tau_C4': 27.829965459147356, 'tau_C5': 58.240687597957766, 'tau_C6': 740.1523149778474}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:15,181] Trial 416 finished with value: 151624.33556808368 and parameters: {'tau_C1': 2.4107720681795612, 'tau_C2': 23.86711054628569, 'tau_C3': 17.168342344590428, 'tau_C4': 50.89128687516447, 'tau_C5': 58.74946706057352, 'tau_C6': 741.9778476412674}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:15,187] Trial 417 finished with value: 156510.80130283153 and parameters: {'tau_C1': 2.6761835727392294, 'tau_C2': 24.99593328409444, 'tau_C3': 18.667254480345687, 'tau_C4': 28.695176907276796, 'tau_C5': 56.22900575970262, 'tau_C6': 788.1834530628217}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:15,193] Tri

[I 2026-04-25 19:15:15,379] Trial 452 finished with value: 145616.85811400614 and parameters: {'tau_C1': 2.0834914162894265, 'tau_C2': 25.62642668087047, 'tau_C3': 16.31134861022326, 'tau_C4': 28.43298927190832, 'tau_C5': 51.56403571248194, 'tau_C6': 754.325060134199}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:15,384] Trial 453 finished with value: 151711.86422394981 and parameters: {'tau_C1': 2.3543926997253095, 'tau_C2': 31.65659457901763, 'tau_C3': 15.336814498821438, 'tau_C4': 30.525699670838343, 'tau_C5': 50.56989524595329, 'tau_C6': 595.9759207423482}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:15,389] Trial 454 finished with value: 145541.47176851996 and parameters: {'tau_C1': 2.0841322301620715, 'tau_C2': 24.775230012560627, 'tau_C3': 16.095031840255096, 'tau_C4': 27.12012966189255, 'tau_C5': 53.855734139418416, 'tau_C6': 665.4939916184383}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:15,396] Trial

[I 2026-04-25 19:15:15,586] Trial 489 finished with value: 152963.5850145153 and parameters: {'tau_C1': 2.4513944188214047, 'tau_C2': 20.991519067649037, 'tau_C3': 18.50370333192092, 'tau_C4': 23.97337314028552, 'tau_C5': 63.67907633806045, 'tau_C6': 332.86382067013085}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:15,593] Trial 490 finished with value: 150364.23854411996 and parameters: {'tau_C1': 2.3260055617593145, 'tau_C2': 20.87165212417105, 'tau_C3': 16.98725603758714, 'tau_C4': 24.73277369973726, 'tau_C5': 58.28890348562528, 'tau_C6': 399.21765629822323}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:15,598] Trial 491 finished with value: 148398.24942767984 and parameters: {'tau_C1': 2.250819995726534, 'tau_C2': 22.595317087519355, 'tau_C3': 18.375453661259613, 'tau_C4': 26.30172828060873, 'tau_C5': 68.44982507149643, 'tau_C6': 499.68630782052577}. Best is trial 328 with value: 144692.99653193346.
[I 2026-04-25 19:15:15,604] Trial

,number,value,datetime_start,datetime_complete,duration,params_tau_C1,params_tau_C2,params_tau_C3,params_tau_C4,params_tau_C5,params_tau_C6,user_attrs_annual_cost,user_attrs_availability,state
0,328,144692.996532,2026-04-25 19:15:14.737898,2026-04-25 19:15:14.741849,0 days 00:00:00.003951,2.085089,21.784357,19.752351,30.930161,68.507317,824.939576,144692.996532,0.960611,COMPLETE
1,342,144768.066443,2026-04-25 19:15:14.801382,2026-04-25 19:15:14.805525,0 days 00:00:00.004143,2.081665,23.614131,19.614383,29.301364,66.639026,830.759895,144768.066443,0.960589,COMPLETE
2,494,144794.136056,2026-04-25 19:15:15.610575,2026-04-25 19:15:15.615676,0 days 00:00:00.005101,2.081872,24.297502,18.447370,26.887219,74.679458,784.801432,144794.136056,0.960640,COMPLETE
3,277,144891.622591,2026-04-25 19:15:14.515199,2026-04-25 19:15:14.519239,0 days 00:00:00.004040,2.081793,20.830419,17.836374,20.617653,71.392770,641.146174,144891.622591,0.960600,COMPLETE
4,498,144973.840788,2026-04-25 19:15:15.633084,2026-04-25 19:15:15.638428,0 days 00:00:00.005344,2.086018,25.358950,19.180469,28.641392,69.291573,748.302188,144973.840788,0.960532,COMPLETE


In [7]:
TOP_K = 5
real_results = []
for _, row in study_df.head(TOP_K).iterrows():
    tau_vector = {c: float(row[f'params_tau_{c}']) for c in COMPONENT_IDS}
    events = run_with_tau(
        tau_vector,
        printer_ids=TEST_PRINTERS,
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    real_score = scalar_objective(events, components_cfg)
    real_results.append({
        'trial': int(row['number']),
        'surrogate_value': float(row['value']),
        'surrogate_annual_cost': float(row['user_attrs_annual_cost']),
        'real_value': float(real_score['value']),
        'real_annual_cost': float(real_score['annual_cost']),
        'real_availability': float(real_score['availability']),
        **{f'tau_{c}': tau_vector[c] for c in COMPONENT_IDS},
    })
real_df = pd.DataFrame(real_results).sort_values('real_value').reset_index(drop=True)
real_df

,trial,surrogate_value,surrogate_annual_cost,real_value,real_annual_cost,real_availability,tau_C1,tau_C2,tau_C3,tau_C4,tau_C5,tau_C6
0,328,144692.996532,144692.996532,1.050000e+10,6.097095e+06,0.0,2.085089,21.784357,19.752351,30.930161,68.507317,824.939576
1,342,144768.066443,144768.066443,1.050000e+10,6.104029e+06,0.0,2.081665,23.614131,19.614383,29.301364,66.639026,830.759895
2,494,144794.136056,144794.136056,1.050000e+10,6.118645e+06,0.0,2.081872,24.297502,18.447370,26.887219,74.679458,784.801432
3,277,144891.622591,144891.622591,1.050000e+10,6.139796e+06,0.0,2.081793,20.830419,17.836374,20.617653,71.392770,641.146174
4,498,144973.840788,144973.840788,1.050000e+10,6.109052e+06,0.0,2.086018,25.358950,19.180469,28.641392,69.291573,748.302188


In [8]:
winner = real_df.iloc[0]
best_tau = {c: float(winner[f'tau_{c}']) for c in COMPONENT_IDS}
payload = {
    'tau_nom_h': best_tau,
    'validated_on': 'test printers (id 85..99)',
    'surrogate_annual_cost': float(winner['surrogate_annual_cost']),
    'real_annual_cost_eur_per_printer_year': float(winner['real_annual_cost']),
    'real_availability': float(winner['real_availability']),
}
with (RESULTS_DIR / 'best_tau_surrogate.yaml').open('w', encoding='utf-8') as handle:
    yaml.safe_dump(payload, handle, sort_keys=False)
print(yaml.safe_dump(payload, sort_keys=False))

tau_nom_h:
  C1: 2.0850890034640184
  C2: 21.784356854347806
  C3: 19.752350802559324
  C4: 30.93016069967772
  C5: 68.50731725938816
  C6: 824.9395757716759
validated_on: test printers (id 85..99)
surrogate_annual_cost: 144692.99653193346
real_annual_cost_eur_per_printer_year: 6097095.35313441
real_availability: 0.0

